[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ddumu/dourado-minguell-eml-mia-um-p1/blob/monte_carlo_on_policy/Entornos_Complejos/monte_carlo_on_policy.ipynb)

# Estudio del Método Tabular de Monte Carlo On-Policy

En este notebook, se codifica y análiza los resultados del método de Monte Carlo On-Policy para el [entorno Toy Text de Gymnasium: Taxi](https://gymnasium.farama.org/environments/toy_text/taxi/).


## El problema del Taxi

El problema del Taxi consiste en navegar un mundo construído por casillas para trasladar pasajeros entre diferentes localizaciones del mapa.

En concreto, el mundo está formado por una cuadrícula 5x5 con 4 ubicaciones distintivamente marcadas por colores: rojo, verde, amarillo y verde. Estas son las zonas de recogida y dejada de pasajeros. Su espacio de movimiento son las siguientes 6 acciones: recoger o dejar a un pasajero y moverse hacia el norte, sur, este u oeste.

El Taxi aparecerá en una posición aleatoria y su misión consiste en recoger al pasajero en una de estas ubicaciones designadas y dejarlo en otra. Recibe recompensas negativas por desplazamientos y por errores al intentar recoger o dejar pasajeros así como refuerzo positivo si consigue dejarlo satisfactoriamente.

## El método de Monte Carlo On-Policy

El Método de Monte Carlo en el contexto de un PDM como el entorno Taxi de Gymnasium, es una técnica de aprendizaje por refuerzo que se fundamenta en la simulación de episodios completos, permitiendo mejorar una política de decisiones de forma iterativa a partir de la experiencia del agente sin necesidad de conocer el modelo del entorno, es decir, sin conocer las probabilidades de transición ni las recompensas asociadas a las mismas. En su variante On-Policy, el método de Monte Carlo mejora la misma política que utiliza para generar cada episodio. Por tanto, ajusta la política en función de los retornos obtenidos bajo su propia experimentación.

## Funciones

### Imports

In [ ]:
import numpy as np
from tqdm import tqdm
import gymnasium as gym
from matplotlib import pyplot as plt

### Gráficos

In [ ]:
def plot(list_stats):
  # Creamos una lista de índices para el eje x
  indices = list(range(len(list_stats)))

  # Creamos el gráfico
  plt.figure(figsize=(6, 3))
  plt.plot(indices, list_stats)

  # Añadimos título y etiquetas
  plt.title('Proporción de recompensas')
  plt.xlabel('Episodio')
  plt.ylabel('Proporción')

  # Mostramos el gráfico
  plt.grid(True)
  plt.show()

In [ ]:
def plot_episodes_length(list_episodes_length):
  # Creamos una lista de índices para el eje x
  indices = list(range(len(list_episodes_length)))

  # Creamos el gráfico
  plt.figure(figsize=(6, 3))
  plt.plot(indices, list_episodes_length)

  # Añadimos título y etiquetas
  plt.title('Tamaño de los Episodios')
  plt.xlabel('Episodio')
  plt.ylabel('Tamaño')

  # Mostramos el gráfico
  plt.grid(True)
  plt.show()

### Monte Carlo

In [ ]:
# Política epsilon-soft. Se usa para el entrenamiento
def random_epsilon_greedy_policy(Q, epsilon, state, nA):
    pi_A = np.ones(nA, dtype=float) * epsilon / nA
    best_action = np.argmax(Q[state])
    pi_A[best_action] += (1.0 - epsilon)
    return pi_A

# Política epsilon-greedy a partir de una epsilon-soft
def epsilon_greedy_policy(Q, epsilon, state, nA):
    pi_A = random_epsilon_greedy_policy(Q, epsilon, state, nA)
    return np.random.choice(np.arange(nA), p=pi_A)

In [ ]:
def monte_carlo_on_policy_all_visit(env, num_episodes=5000, epsilon=0.4, decay=False, discount_factor=1):
    # Matriz de valores  Q
    nA = env.action_space.n
    Q = np.zeros([env.observation_space.n, nA])

    # Número de visitas. Vamos a realizar la versión incremental.
    n_visits = np.zeros([env.observation_space.n, env.action_space.n])

    # Para mostrar la evolución en el terminal y algún dato que mostrar
    stats = 0.0
    list_stats = [stats]
    list_episodes_length = []
    step_display = num_episodes / 10

    for t in tqdm(range(num_episodes)):
        state, info = env.reset(seed=100)
        done = False
        episode = []
        result_sum = 0.0

        while not done:
            if decay:
                epsilon = min(1.0, 1000.0 / (t + 1))

            # Decidir acción mediante política epsilon greedy
            action = epsilon_greedy_policy(Q, epsilon, state, nA)

            # Tomar la acción generando un nuevo estado y recompensa
            new_state, reward, terminated, truncated, info = env.step(action)

            # Marcar el fin del episodio de forma condicional
            done = terminated or truncated

            # Guardar el retorno para la acción tomada en el estado y tiempo actual
            episode.append((state, action, reward))

            # Actualizar el estado
            state = new_state

        for e in range(len(episode) - 1, -1, -1):
            # Obtener estado y acción en t
            state, action, reward = episode[e]

            # Actualizar la recompensa en t
            result_sum = result_sum * discount_factor + reward

            # Actualizar el número de visitas al estado-acción
            n_visits[state, action] += 1.0

            # Calcular alpha para actualización incremental
            alpha = 1.0 / n_visits[state, action]

            # Actualizar q(a|s) de forma incremental
            Q[state, action] += alpha * (result_sum - Q[state, action])

        # Guardamos datos sobre la evolución. Promedio de recompensas
        stats += result_sum
        list_stats.append(stats / (t + 1))
        list_episodes_length.append(len(episode))

        # Para mostrar la evolución.  Comentar si no se quiere mostrar
        if t % step_display == 0 and t != 0:
            print(f"success: {stats / t}, epsilon: {epsilon}")

    return Q, list_stats, list_episodes_length

### Gymnasium

In [ ]:
env = gym.make('Taxi-v3')

## Experimentación

In [ ]:
n_episodes = 5000
epsilon = 0.4
Q, list_stats, list_episodes_length = monte_carlo_on_policy_all_visit(env, n_episodes, epsilon)

In [ ]:
plot(list_stats)

In [ ]:
plot_episodes_length(list_episodes_length)